In [ ]:
pip install xarray netCDF4 rasterio numpy tqdm


In [ ]:
#For NetCDF(.nc) files

import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
import zipfile
from tqdm import tqdm

# Input and output file paths
input_nc_file = "/content/RF25_ind2025_rfp25.nc"
output_zip_file = "NCdaily_rainfall_geotiffs2025.zip"
output_folder = "/content/NCdaily_geotiffs2025"

# Create output directory
os.makedirs(output_folder, exist_ok=True)
# Open the NetCDF file
dataset = xr.open_dataset(input_nc_file)

# Extract required variables
rainfall_data = dataset['RAINFALL']  # Replace 'rainfall' with the actual variable name in your dataset
lon = dataset['LONGITUDE'].values  # Replace 'longitude' with the actual name in your dataset
lat = dataset['LATITUDE'].values  # Replace 'latitude' with the actual name in your dataset
days = rainfall_data.shape[0]

# Transform for GeoTIFF
transform = from_origin(lon[0] - 0.125, lat[-1] + 0.125, 0.25, 0.25)

# Save each day as GeoTIFF
print(f"Saving GeoTIFF files to {output_folder}...")
for day in tqdm(range(days), desc="Processing days"):
    day_data = rainfall_data[day, :, :].values
    day_data = np.flipud(day_data)  # Flip to match GeoTIFF orientation

    output_file = os.path.join(output_folder, f"day{day + 1}.tif")

    # Save the GeoTIFF
    with rasterio.open(
        output_file,
        'w',
        driver='GTiff',
        height=day_data.shape[0],
        width=day_data.shape[1],
        count=1,
        dtype=day_data.dtype,
        crs="EPSG:4326",
        transform=transform
    ) as dst:
        dst.write(day_data, 1)

# Compress GeoTIFF files into a ZIP archive
print(f"Compressing GeoTIFF files into {output_zip_file}...")
with zipfile.ZipFile(output_zip_file, 'w') as zipf:
    for root, _, files in os.walk(output_folder):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

# Cleanup
print(f"GeoTIFF files compressed into {output_zip_file}.")


Saving GeoTIFF files to /content/NCdaily_geotiffs2025...


Processing days: 100%|██████████| 365/365 [00:01<00:00, 295.62it/s]


Compressing GeoTIFF files into NCdaily_rainfall_geotiffs2025.zip...
GeoTIFF files compressed into NCdaily_rainfall_geotiffs2025.zip.
